# HoloKai V4 — OpenMed privacy-filter baseline

不微调、不在 V4 上调参。上传配套 `privacy_filter_v4_assets.zip`，选择 **GPU**，依次运行。

本 notebook 尚未完成 GPU 实测；最终结果以 Colab 成功运行的输出为准。

## V2 检查后的更新
- 用合法 BIOES 路径的 constrained Viterbi 替代逐位置 argmax 后修补；没有拟合转移参数，没有按 V4 分数选择解码器。
- 长文本重叠窗口采用中心加权 logits 融合，再统一解码。T4 不支持 BF16 时用 FP32，避免 FP16 数值风险。
- 保存实际 id2label、完整环境版本和 argmax 原始片段，便于复核。
- 标签不匹配限制仍然存在：本配置仅向 PERSON、ADDRESS、ID 生成预测。完整 11 类召回率理论上最多 360/1320=27.27%，F1 理想上限 42.86%；不能将其与全类别临床 NER 的总分作为同任务能力排名。

## 评测规则（运行前固定）
- 原始 V4：648 条、1,320 个实体、11 类，校验文件 SHA256。
- 与旧 GLiNER 一致：原始词元的 inclusive start/end 与标签严格匹配。
- 完整 11 类：姓名统一映射到 PERSON（role-blind fallback）；PROVIDER 及其他不支持类别仍留在 gold 中计漏检。
- 共同映射子集：仅 LOCATION_ADDRESS、UNIQUE_ID。这是基于规则的近似对应，不能称作完全相同的原生标签。
- 姓名诊断：PERSON/PROVIDER 合并为 NAME；不代表能够识别医疗角色。
- SEX/GENDER_IDENTITY 不映射。模型的公司名也不直接映射为医疗机构。
- 仅合并相邻姓名片段；地址合并允许空白/逗号间隔，且必须有 street_address。编号映射名单在 frozen_protocol.json 中记录。
- 非原始词元边界的预测仍计 FP，不做自动扩展。范围外 PII 另存 raw_char_spans，不计入 V4 标签范围内的 FP。
- 模型版本在首次运行解析为不可变 commit，保存环境版本；若要复现，使用该 commit。

来源：[模型卡](https://huggingface.co/OpenMed/privacy-filter-nemotron-v2)、用户原始 V4 和旧 GLiNER 预测。

## 1. 安装依赖
在新 GPU runtime 中先运行本单元格。

In [ ]:
%pip install -q -U transformers accelerate huggingface_hub pandas

## 2. 上传并解压配套数据包

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()  # 选择 privacy_filter_v4_assets.zip
archive = next(k for k in uploaded if k.endswith('.zip'))
root = Path('/content/privacy_filter_v4')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
        if not (root / member).resolve().is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(root)
os.chdir(root)
print('Ready:', root)

## 3. 固定本次输出目录并运行
默认将输出保存在 Google Drive。首次运行会下载模型；推理每 25 条输出一次进度。不要根据 V4 分数修改映射或阈值。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = '/content/drive/MyDrive/Holokai_Project/week9/privacy_filter_nemotron_v2_v4_baseline_reviewed_v2'
# 复现时填入上次 frozen_protocol.json 中的 revision；首次使用 main。
REVISION = 'main'
import subprocess, sys
subprocess.run([sys.executable, '-u', 'baseline.py', '--data', 'data/v4.json',
                '--output', OUTPUT, '--revision', REVISION], check=True)

## 4. 查看结果，并在相同范围重算旧 GLiNER
旧 GLiNER 使用已有固定阈值预测，不重新运行、不重新选阈值。

In [ ]:
import json
import pandas as pd
from baseline import LABELS, COMMON, score
v4 = json.loads(Path('data/v4.json').read_text())
previous = json.loads(Path('data/gliner_fixed_predictions.json').read_text())
assert len(previous) == len(v4)
for i,(old,row) in enumerate(zip(previous,v4)):
    assert old['sample_index'] == i
    assert old['tokenized_text'] == row['tokenized_text']
    assert old['gold_ner'] == row['ner']
gold = [r['ner'] for r in v4]
old_pred = [[[e['start'],e['end'],e['label']] for e in r['predicted_entities']] for r in previous]
current = json.loads((Path(OUTPUT)/'metrics.json').read_text())
comparison = []
for scope, labels in [('full_11',LABELS),('mapped_subset_2',COMMON)]:
    for model, result in [('OpenMed',current[scope]),('GLiNER fixed',score(gold,old_pred,labels))]:
        comparison.append(dict(model=model,scope=scope,**result['overall']))
ng = [[[a,b,'NAME'] for a,b,l in g if l in {'name','name medical professional'}] for g in gold]
np = [[[a,b,'NAME'] for a,b,l in g if l in {'name','name medical professional'}] for g in old_pred]
for model,result in [('OpenMed',current['merged_name_diagnostic']),('GLiNER fixed',score(ng,np,['NAME']))]:
    comparison.append(dict(model=model,scope='merged_name_diagnostic',**result['overall']))
table = pd.DataFrame(comparison)
table.to_csv(Path(OUTPUT)/'comparison.csv',index=False)
display(table.style.format({k:'{:.2%}' for k in ['precision','recall','f1']}))
display(pd.read_csv(Path(OUTPUT)/'per_label.csv'))

## 5. 下载结果
包含原始预测、映射后预测、指标、逐类表、对照表和固定协议。不要把不支持类别的零分解释为同类识别能力对比。

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/privacy_filter_v4_results','zip',OUTPUT)
files.download(archive)